# Step 5: Static stay table

One row per stay, one schema for all three sites, built from the assembled tables by `static.py`. This is the table that carries age, sex, ethnicity, weight, length of stay, outcome, diagnosis group and PIM, and it fixes the set of stays that every later step uses.

**What this notebook does**
1. Records a baseline of the static tables saved earlier (stays, columns and a fingerprint of the stay keys), before anything is overwritten.
2. Rebuilds the three site tables from the assembled parquet files with `S.build_all`, so nothing depends on an older file.
3. Compares the rebuild with the baseline: same stays, same columns, same fingerprint.
4. Checks the pooled columns (coverage, categories, PIM by year) and saves the tables and a counts-only report.

The rules (PICANet first, weight valid 0.3 to 200 kg, LOS from PICANet else outcomes, stays with two PICANet rows excluded, ethnicity from PICANet with the RBH demographics fallback, no free text) are in the header of `static.py`. Only counts go in the report. The static tables themselves stay in `derived/`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import inventory as I, manifest as M, diagnostics as D, static as S

## 1. Directories and baseline
`static_dir` holds the static tables saved earlier. The first time this notebook runs, the cell below writes their stay counts, column counts, column names and a fingerprint of the stays to `reports/Static_counts_baseline.csv`. After that the file is only read, never rewritten, so a later run compares against the original saved tables and not against its own earlier rebuild.

The tables saved earlier (17 November) were built before the assembly rebuild and before the current `static.py`, and they have no `stay_key` column. The fingerprint therefore identifies a stay by patient and admission time (`patient_id|t0`), which both the old and the rebuilt tables have. The fingerprint is the number of stays plus a 12-character hash of the sorted keys, so it shows that the stays are the same ones, not just that the count matches. The column names are stored too, so the comparison can list exactly which columns are new.

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")  # keep whatever the assemble notebook uses
derived = open_picu_dir / "derived"
out_dir = derived / "assembled"
static_dir = derived / "static"
reports_dir = open_picu_dir / "reports"
static_dir.mkdir(exist_ok=True)
baseline_path = reports_dir / "Static_counts_baseline.csv"


stay_fingerprint = S.stay_fingerprint


if not baseline_path.exists():
    rows = []
    for name in ["RBH", "SMH", "GOSH", "pooled"]:
        df = pd.read_parquet(static_dir / f"{name}_static.parquet")
        rows.append({"table": name, "rows": len(df), "columns": df.shape[1],
                     "fingerprint": stay_fingerprint(df)["hash"], "column names": "; ".join(df.columns)})
    pd.DataFrame(rows).to_csv(baseline_path, index=False)
baseline = pd.read_csv(baseline_path)
baseline[["table", "rows", "columns", "fingerprint"]]

## 2. Rebuild the static tables from the assembled files
`S.build_all` reads the assembled parquet files in `derived/assembled/` and builds one row per stay for each site, using the rules at the top of `static.py`. It prints, per site, the number of stays, how many were excluded because PICANet had two conflicting rows, which source the weight and length of stay came from, and the outcome counts. Nothing is read from the old static tables here, so the result cannot depend on an earlier run.

`S.pool` then stacks the common columns of the three sites into the pooled table. Both are only held in memory at this point; saving is the last section, after the comparison.

Expected from the earlier run: RBH 4,764 stays (1 excluded), SMH 3,697 (2 excluded), GOSH 9,671 (0 excluded), pooled 18,132.

In [ ]:
static = S.build_all(out_dir)
pooled = S.pool(static)
len(pooled)

## 3. Compare with the saved tables
The rebuilt tables are compared with the baseline recorded above, for each site and for the pooled table:
- **same rows**: the number of stays is unchanged.
- **same stays**: the fingerprint of `patient_id|t0` is unchanged, so they are the same stays and not just the same count.
- **columns added / removed**: which columns differ from the 17 November tables. Only `stay_key` should be added. Anything else listed needs explaining before we go on.

This shows that the rebuild from the freshly assembled tables gives the same stays as the tables the later steps were built on. If every row reads True, nothing downstream changes.

In [ ]:
check = S.compare_with_baseline(static, pooled, baseline)
check

## 4. Compare the values, not just the stays
Section 3 showed that the rebuild contains the same stays as the saved tables. This checks that the values in them are the same too. For each table and each column that both versions have, `S.compare_values` matches the stays on `patient_id|t0` and counts the cells that differ. Of those, it counts how many were missing before and are filled now, and the reverse. Only counts are shown, and columns with no differences are left out, so an empty table means every shared column is identical.

The saved tables were made on 17 November, before the RBH ethnicity fallback from demographics (the new `ethnicity_source` column), so I expect differences in RBH `ethnicity` and the pooled `ethnicity`, with about 1,578 cells missing before and filled now. Any other column listed, or any cell filled before and missing now, needs explaining before the later steps are trusted.

In [ ]:
changes = S.compare_values(static, pooled, static_dir)
changes

## 5. What changed in ethnicity
Section 4 found that `ethnicity` is the only column whose values differ from the saved tables. For each site, this lists which old value became which new value, with the number of stays (counts under 10 shown as `<10`). The expected causes are the fold of "Unknown" into "Not stated" (all sites) and, for RBH, the fill from demographics where PICANet says "Not stated" or has no record. Anything that is neither of these needs explaining.

In [ ]:
for site in ["RBH", "SMH", "GOSH"]:
    print(site)
    print(I.df_text(S.column_changes(static, static_dir, site), index=False))
    print()

## 6. Checks on the pooled columns
Non-null % and number of distinct values of every pooled column by site, to confirm that each column is filled where the site can supply it. Then the category counts of the columns that will be released or used for grouping (sex, diagnosis group), and ethnicity for reference only, since it is not released. Counts under 10 are shown as `<10`.

In [ ]:
S.coverage(static)

In [ ]:
sex_t = S.category_table(static, "sex")
dx_t = S.category_table(static, "diagnosis_group")
eth_t = S.category_table(static, "ethnicity")

## 7. PIM by site and year
A change of PIM version at a site would show as a step in the percentiles in one admission year. GOSH records the column only as `PIM`, with no version, so a step there is the thing to look for.

In [ ]:
S.pim_by_year(static)

## 8. Short stays
How many stays have a length of stay of 6 hours or less, by site. These stays cannot have many hourly rows, so this is the size of the group that will be thin in any hourly table. Counts under 10 are shown as `<10`.

In [ ]:
S.short_stays(static)

## 9. Save

In [ ]:
for site, df in static.items():
    df.to_parquet(static_dir / f"{site}_static.parquet", index=False)
pooled.to_parquet(static_dir / "pooled_static.parquet", index=False)


def masked(t):
    return t.map(lambda v: "-" if v == 0 else ("<10" if v < 10 else str(v)))


D.write_report({
    "Rebuild against the saved tables": check,
    "Cells that differ from the saved tables": changes,
    "Rows per site": pooled.groupby("site").size().to_frame("stays"),
    "Sex": masked(sex_t),
    "Diagnosis group": masked(dx_t),
    "Ethnicity (not released)": masked(eth_t),
    "Short stays": S.short_stays(static),
}, reports_dir / f"Static_{pd.Timestamp.today().date()}.md", title="Static tables")
pooled.groupby("site").size()